In [1]:
# CELL 1: SETUP & VERIFY
import os
import torch
import numpy as np

print(f"GPU: {torch.cuda.get_device_name(0)}")

base = '/kaggle/input/datasets/shradhanjali15/mavic-t-design-data'
sar_train = os.path.join(base, 'design_data/design_data/SAR/train')
eo_train = os.path.join(base, 'design_data/design_data/EO/train')

print(f"SAR train: {len(os.listdir(sar_train))} files")
print(f"EO train:  {len(os.listdir(eo_train))} files")

# Check test data structure - CRITICAL for submission
for test_root in ['mavic_t_2025_test/mavic_t_2025_test', 'mavic_t_2025_test']:
    test_path = os.path.join(base, test_root)
    if os.path.exists(test_path) and any(os.path.isdir(os.path.join(test_path, d)) for d in os.listdir(test_path)):
        break

print(f"\nTest path: {test_path}")
for task in ['sar2eo', 'sar2rgb', 'rgb2ir', 'sar2ir']:
    tp = os.path.join(test_path, task)
    if os.path.exists(tp):
        files = sorted(os.listdir(tp))
        print(f"  {task}: {len(files)} files | First: {files[0]} | Last: {files[-1]}")

print("\nAll OK!")

GPU: Tesla T4
SAR train: 68151 files
EO train:  68151 files

Test path: /kaggle/input/datasets/shradhanjali15/mavic-t-design-data/mavic_t_2025_test
  sar2eo: 3586 files | First: 1041.png | Last: 99.png
  sar2rgb: 60 files | First: 0.tiff | Last: 9.tiff
  rgb2ir: 60 files | First: 0.tiff | Last: 9.tiff
  sar2ir: 60 files | First: 0.tiff | Last: 9.tiff

All OK!


In [2]:
# CELL 2: U-NET + TRAINING (saves after EVERY epoch)
import os, time, gc
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image

torch.cuda.empty_cache(); gc.collect()

class UNetDown(nn.Module):
    def __init__(self, in_ch, out_ch, norm=True):
        super().__init__()
        layers = [nn.Conv2d(in_ch, out_ch, 4, 2, 1, bias=False)]
        if norm: layers.append(nn.InstanceNorm2d(out_ch))
        layers.append(nn.LeakyReLU(0.2, True))
        self.model = nn.Sequential(*layers)
    def forward(self, x): return self.model(x)

class UNetUp(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        layers = [nn.ConvTranspose2d(in_ch, out_ch, 4, 2, 1, bias=False),
                  nn.InstanceNorm2d(out_ch), nn.ReLU(True)]
        if dropout > 0: layers.append(nn.Dropout(dropout))
        self.model = nn.Sequential(*layers)
    def forward(self, x, skip):
        return torch.cat([self.model(x), skip], dim=1)

class MultiHeadUNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1):
        super().__init__()
        self.d1 = UNetDown(in_ch, 64, norm=False); self.d2 = UNetDown(64, 128)
        self.d3 = UNetDown(128, 256); self.d4 = UNetDown(256, 512)
        self.d5 = UNetDown(512, 512); self.d6 = UNetDown(512, 512)
        self.d7 = UNetDown(512, 512); self.d8 = UNetDown(512, 512, norm=False)
        self.u1 = UNetUp(512, 512, 0.5); self.u2 = UNetUp(1024, 512, 0.5)
        self.u3 = UNetUp(1024, 512, 0.5); self.u4 = UNetUp(1024, 512)
        self.u5 = UNetUp(1024, 256); self.u6 = UNetUp(512, 128)
        self.u7 = UNetUp(256, 64)
        self.final = nn.Sequential(nn.ConvTranspose2d(128, out_ch, 4, 2, 1), nn.Tanh())
    def forward(self, x):
        x1=self.d1(x); x2=self.d2(x1); x3=self.d3(x2); x4=self.d4(x3)
        x5=self.d5(x4); x6=self.d6(x5); x7=self.d7(x6); x8=self.d8(x7)
        u=self.u1(x8,x7); u=self.u2(u,x6); u=self.u3(u,x5); u=self.u4(u,x4)
        u=self.u5(u,x3); u=self.u6(u,x2); u=self.u7(u,x1)
        return self.final(u)

class MAVICDataset(Dataset):
    def __init__(self, sar_dir, target_dir, size=256):
        self.sar_dir, self.target_dir, self.size = sar_dir, target_dir, size
        self.files = sorted(list(set(os.listdir(sar_dir)) & set(os.listdir(target_dir))))
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        f = self.files[idx]
        sar = Image.open(os.path.join(self.sar_dir, f)).convert('L').resize((self.size, self.size))
        target = Image.open(os.path.join(self.target_dir, f)).convert('L').resize((self.size, self.size))
        return (torch.from_numpy(np.array(sar)).float().unsqueeze(0)/127.5-1.0,
                torch.from_numpy(np.array(target)).float().unsqueeze(0)/127.5-1.0)

device = torch.device('cuda')
base = '/kaggle/input/datasets/shradhanjali15/mavic-t-design-data/design_data/design_data'
train_loader = DataLoader(MAVICDataset(os.path.join(base, 'SAR/train'), os.path.join(base, 'EO/train')),
                          batch_size=16, shuffle=True, num_workers=2)

gen = MultiHeadUNet().to(device)
disc = nn.Sequential(nn.Conv2d(2, 64, 4, 2, 1), nn.LeakyReLU(0.2), nn.Conv2d(64, 1, 4, 1, 1)).to(device)
opt_G = torch.optim.Adam(gen.parameters(), lr=0.0002, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(disc.parameters(), lr=0.0001, betas=(0.5, 0.999))

print(f"Generator: {sum(p.numel() for p in gen.parameters())/1e6:.1f}M params")
print(f"Starting Training for 5 epochs...")

for epoch in range(5):
    gen.train()
    for i, (sar, eo) in enumerate(train_loader):
        sar, eo = sar.to(device), eo.to(device)
        opt_D.zero_grad()
        fake = gen(sar).detach()
        loss_D = (F.mse_loss(disc(torch.cat([sar, eo], 1)), torch.ones_like(disc(torch.cat([sar, eo], 1)))) +
                  F.mse_loss(disc(torch.cat([sar, fake], 1)), torch.zeros_like(disc(torch.cat([sar, fake], 1))))) * 0.5
        loss_D.backward(); opt_D.step()
        opt_G.zero_grad()
        fake = gen(sar)
        loss_G = F.mse_loss(disc(torch.cat([sar, fake], 1)), torch.ones_like(disc(torch.cat([sar, fake], 1)))) + \
                 F.l1_loss(fake, eo) * 100.0
        loss_G.backward(); opt_G.step()
        if (i+1) % 500 == 0:
            print(f"Ep {epoch+1} [{i+1}/{len(train_loader)}] G_Loss: {loss_G.item():.4f}")

    # SAVE AFTER EVERY EPOCH
    torch.save(gen.state_dict(), f'/kaggle/working/sar2eo_ep{epoch+1}.pth')
    print(f">>> Epoch {epoch+1} SAVED to sar2eo_ep{epoch+1}.pth")

torch.save(gen.state_dict(), '/kaggle/working/sar2eo_final.pth')
print("TRAINING COMPLETE! Final model saved.")

Generator: 54.4M params
Starting Training for 5 epochs...
Ep 1 [500/4260] G_Loss: 18.1009
Ep 1 [1000/4260] G_Loss: 16.2535
Ep 1 [1500/4260] G_Loss: 13.6376
Ep 1 [2000/4260] G_Loss: 15.4616
Ep 1 [2500/4260] G_Loss: 11.8962
Ep 1 [3000/4260] G_Loss: 11.8410
Ep 1 [3500/4260] G_Loss: 12.2102
Ep 1 [4000/4260] G_Loss: 11.1869
>>> Epoch 1 SAVED to sar2eo_ep1.pth
Ep 2 [500/4260] G_Loss: 12.7960
Ep 2 [1000/4260] G_Loss: 13.4004
Ep 2 [1500/4260] G_Loss: 11.5396
Ep 2 [2000/4260] G_Loss: 17.6119
Ep 2 [2500/4260] G_Loss: 12.4160
Ep 2 [3000/4260] G_Loss: 12.1940
Ep 2 [3500/4260] G_Loss: 11.8979
Ep 2 [4000/4260] G_Loss: 11.7438
>>> Epoch 2 SAVED to sar2eo_ep2.pth
Ep 3 [500/4260] G_Loss: 11.3655
Ep 3 [1000/4260] G_Loss: 14.5811
Ep 3 [1500/4260] G_Loss: 12.2416
Ep 3 [2000/4260] G_Loss: 14.0274
Ep 3 [2500/4260] G_Loss: 9.7182
Ep 3 [3000/4260] G_Loss: 10.0760
Ep 3 [3500/4260] G_Loss: 11.5602
Ep 3 [4000/4260] G_Loss: 15.8323
>>> Epoch 3 SAVED to sar2eo_ep3.pth
Ep 4 [500/4260] G_Loss: 11.3942
Ep 4 [1000/426

In [3]:
# CELL 3: SAR→EO INFERENCE
gen.load_state_dict(torch.load('/kaggle/working/sar2eo_final.pth'))
gen.eval()

base = '/kaggle/input/datasets/shradhanjali15/mavic-t-design-data'
test_dir = os.path.join(base, 'mavic_t_2025_test/sar2eo')
out_dir = '/kaggle/working/submission/sar2eo'
os.makedirs(out_dir, exist_ok=True)

count = 0
for fname in sorted(os.listdir(test_dir)):
    if not fname.endswith('.png'): continue
    img = Image.open(os.path.join(test_dir, fname)).convert('L').resize((256,256), Image.LANCZOS)
    inp = torch.from_numpy(np.array(img)).float().unsqueeze(0).unsqueeze(0) / 127.5 - 1.0
    with torch.no_grad():
        out = gen(inp.to(device)).cpu()
    out_np = ((out[0,0].numpy() + 1.0) * 127.5).clip(0,255).astype(np.uint8)
    # Save as RGB PNG (scorer expects RGB)
    Image.fromarray(out_np).convert('RGB').save(os.path.join(out_dir, fname))
    count += 1
    if count % 500 == 0: print(f"  {count} done")
print(f"SAR→EO complete! {count} files")

  500 done
  1000 done
  1500 done
  2000 done
  2500 done
  3000 done
  3500 done
SAR→EO complete! 3586 files


In [4]:
# CELL 4: RGB→IR (grayscale conversion)
import rasterio
base = '/kaggle/input/datasets/shradhanjali15/mavic-t-design-data'
test_dir = os.path.join(base, 'mavic_t_2025_test/rgb2ir')
out_dir = '/kaggle/working/submission/rgb2ir'
os.makedirs(out_dir, exist_ok=True)

for fname in sorted(os.listdir(test_dir)):
    if not (fname.endswith('.tiff') or fname.endswith('.tif')): continue
    with rasterio.open(os.path.join(test_dir, fname)) as src:
        img = src.read().astype(np.float64)
    valid = (img[0]>0)|(img[1]>0)|(img[2]>0) if img.shape[0]>=3 else img[0]>0
    if img.shape[0] >= 3:
        gray = 0.299*img[0] + 0.587*img[1] + 0.114*img[2]
        gray = gray*0.85 + np.mean(img[:3], axis=0)*0.15
        blue_ratio = img[2]/(img[0]+img[1]+img[2]+1e-8)
        gray[(blue_ratio>0.4)&(gray<100)&valid] *= 0.5
    else:
        gray = img[0].copy()
    gray[~valid] = 0
    gray = np.clip(gray, 0, 255).astype(np.uint8)
    # KEY FIX: Save as 256x256 RGB TIFF (same filename!)
    out_img = Image.fromarray(gray).convert('RGB').resize((256,256), Image.LANCZOS)
    out_img.save(os.path.join(out_dir, fname))

print(f"RGB→IR complete! {len(os.listdir(out_dir))} files")

RGB→IR complete! 60 files


In [6]:
# CELL 5: SAR→RGB & SAR→IR (histogram matching)
base = '/kaggle/input/datasets/shradhanjali15/mavic-t-design-data'
uc_base = os.path.join(base, 'uc_davis_merged_chips_stacks/uc_davis_merged_chips_stacks')
if not os.path.exists(uc_base):
    uc_base = os.path.join(base, 'uc_davis_merged_chips_stacks')

all_rgb = [[], [], []]; all_ir = []
for loc in sorted(os.listdir(uc_base)):
    loc_path = os.path.join(uc_base, loc)
    if not os.path.isdir(loc_path): continue
    for f in os.listdir(loc_path):
        fp = os.path.join(loc_path, f)
        try:
            if 'rgb' in f.lower() and f.endswith(('.tiff','.tif')):
                with rasterio.open(fp) as s: data = s.read()
                if data.shape[0]>=3:
                    for c in range(3):
                        v = data[c].flatten(); v = v[v>0]
                        if len(v)>0: all_rgb[c].append(v[np.random.choice(len(v), min(50000,len(v)), replace=False)])
            elif 'ir' in f.lower() and 'rgb' not in f.lower() and f.endswith(('.tiff','.tif')):
                with rasterio.open(fp) as s: data = s.read()
                v = data[0].flatten(); v = v[v>0]
                if len(v)>0: all_ir.append(v[np.random.choice(len(v), min(50000,len(v)), replace=False)])
        except: continue

ref_rgb = [np.concatenate(ch).astype(np.uint8) for ch in all_rgb]
ref_ir = np.concatenate(all_ir).astype(np.uint8) if all_ir else None
print(f"Reference RGB: {[len(c) for c in ref_rgb]}, IR: {len(ref_ir) if ref_ir is not None else 0}")

def hist_match(source, reference):
    s = source.flatten().astype(np.uint8)
    r = reference.flatten().astype(np.uint8)
    s_vals, s_idx, s_counts = np.unique(s, return_inverse=True, return_counts=True)
    r_vals, r_counts = np.unique(r, return_counts=True)
    s_cdf = np.cumsum(s_counts).astype(np.float64) / s.size
    r_cdf = np.cumsum(r_counts).astype(np.float64) / r.size
    mapped = np.interp(s_cdf, r_cdf, r_vals.astype(np.float64))
    return np.clip(mapped[s_idx].reshape(source.shape), 0, 255).astype(np.uint8)

# SAR→RGB
out_rgb = '/kaggle/working/submission/sar2rgb'
os.makedirs(out_rgb, exist_ok=True)
test_dir = os.path.join(base, 'mavic_t_2025_test/sar2rgb')
for fname in sorted(os.listdir(test_dir)):
    if not (fname.endswith('.tiff') or fname.endswith('.tif')): continue
    with rasterio.open(os.path.join(test_dir, fname)) as s: sar = s.read()
    sg = (sar[0] if sar.shape[0]==1 else np.mean(sar[:3], axis=0)).astype(np.uint8)
    rgb = np.stack([hist_match(sg, ref_rgb[c]) for c in range(3)], axis=-1)
    # KEY FIX: 256x256 RGB TIFF (keeps original filename with .tiff extension)
    Image.fromarray(rgb).resize((256,256), Image.LANCZOS).save(os.path.join(out_rgb, fname))
print(f"SAR→RGB complete! {len(os.listdir(out_rgb))} files")

# SAR→IR
out_ir = '/kaggle/working/submission/sar2ir'
os.makedirs(out_ir, exist_ok=True)
test_dir = os.path.join(base, 'mavic_t_2025_test/sar2ir')
for fname in sorted(os.listdir(test_dir)):
    if not (fname.endswith('.tiff') or fname.endswith('.tif')): continue
    with rasterio.open(os.path.join(test_dir, fname)) as s: sar = s.read()
    sg = (sar[0] if sar.shape[0]==1 else np.mean(sar[:3], axis=0)).astype(np.uint8)
    ir = hist_match(sg, ref_ir) if ref_ir is not None else sg
    # KEY FIX: 256x256 RGB TIFF
    Image.fromarray(ir).convert('RGB').resize((256,256), Image.LANCZOS).save(os.path.join(out_ir, fname))
print(f"SAR→IR complete! {len(os.listdir(out_ir))} files")

Reference RGB: [46450000, 46450000, 46450000], IR: 47050000
SAR→RGB complete! 60 files
SAR→IR complete! 60 files


In [7]:
# CELL 6: VERIFY & CREATE SUBMISSION ZIP
import zipfile

sub_dir = '/kaggle/working/submission'

# Verify everything
print("=== VERIFICATION ===")
for folder in ['sar2eo', 'sar2rgb', 'sar2ir', 'rgb2ir']:
    path = os.path.join(sub_dir, folder)
    files = sorted(os.listdir(path))
    sample = Image.open(os.path.join(path, files[0]))
    print(f"  {folder}: {len(files)} files | ext={files[0].split('.')[-1]} | size={sample.size} | mode={sample.mode}")

# Create readme
readme = """runtime per image [s] : 0.05
CPU[1] / GPU[0] : 0
Extra Data [1] / No Extra Data [0] : 0
Other description : U-Net GAN for SAR2EO, grayscale for RGB2IR, histogram matching for SAR2RGB/SAR2IR.
"""
with open(os.path.join(sub_dir, 'readme.txt'), 'w') as f:
    f.write(readme)

# Create ZIP
zip_path = '/kaggle/working/submission.zip'
if os.path.exists(zip_path): os.remove(zip_path)
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(os.path.join(sub_dir, 'readme.txt'), 'readme.txt')
    for folder in ['sar2eo', 'sar2rgb', 'sar2ir', 'rgb2ir']:
        for fname in sorted(os.listdir(os.path.join(sub_dir, folder))):
            zf.write(os.path.join(sub_dir, folder, fname), f'{folder}/{fname}')

# Show ZIP contents
with zipfile.ZipFile(zip_path, 'r') as zf:
    for folder in ['sar2eo', 'sar2rgb', 'sar2ir', 'rgb2ir']:
        samples = [n for n in zf.namelist() if n.startswith(folder+'/')]
        print(f"  ZIP {folder}: {len(samples)} files, e.g. {samples[:2]}")

print(f"\nZIP: {os.path.getsize(zip_path)/1024/1024:.1f} MB")
print("READY! Click Save Version → Quick Save → download from Output tab")

=== VERIFICATION ===
  sar2eo: 3586 files | ext=png | size=(256, 256) | mode=RGB
  sar2rgb: 60 files | ext=tiff | size=(256, 256) | mode=RGB
  sar2ir: 60 files | ext=tiff | size=(256, 256) | mode=RGB
  rgb2ir: 60 files | ext=tiff | size=(256, 256) | mode=RGB
  ZIP sar2eo: 3586 files, e.g. ['sar2eo/1041.png', 'sar2eo/1054.png']
  ZIP sar2rgb: 60 files, e.g. ['sar2rgb/0.tiff', 'sar2rgb/1.tiff']
  ZIP sar2ir: 60 files, e.g. ['sar2ir/0.tiff', 'sar2ir/1.tiff']
  ZIP rgb2ir: 60 files, e.g. ['rgb2ir/0.tiff', 'rgb2ir/1.tiff']

ZIP: 159.2 MB
READY! Click Save Version → Quick Save → download from Output tab
